In [1]:
import sys
import os
import pandas as pd

# Add project root to Python path to find the 'src' directory
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

Added project root to sys.path: c:\Users\peera\Desktop\DroughtLSTM_oneday


In [8]:
from google.cloud import storage

In [9]:
gcs_client = storage.Client.create_anonymous_client()

In [12]:
xr.open_zarr('gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr')

FileNotFoundError: No such file or directory: 'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr'

In [17]:
import xarray as xr
from dask.diagnostics import ProgressBar
from pathlib import Path # For checking file size locally


# --- IMPORTANT NOTES FOR LOCAL VS CODE ---
# 1. No 'google.colab.auth' is needed; authentication is handled by 'gcloud auth application-default login' run in your terminal.
# 2. No '!pip install' commands are needed in the script itself; packages are installed once into your virtual environment.
# 3. Output will be saved directly to your local file system.
# ---

# 1. Load the public Zarr dataset anonymously
# 'storage_options={"token": "cloud"}' tells gcsfs to use the Application Default Credentials
# you set up with 'gcloud auth application-default login'.
print("🚀 Attempting to load dataset from Google Cloud Storage...")
ds = xr.open_zarr(
    "gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr",
)
print("✅ Dataset loaded successfully (metadata).")


# 2. Define variables to keep
vars_to_keep = [
    'total_precipitation_6hr',
    '2m_temperature', '2m_dewpoint_temperature',
    'surface_pressure', 'mean_sea_level_pressure',
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'u_component_of_wind', 'v_component_of_wind',
    'specific_humidity', 'relative_humidity',
    'total_column_water_vapour', 'total_cloud_cover',
    'mean_surface_net_short_wave_radiation_flux',
    'mean_surface_latent_heat_flux',
    'vertical_velocity', 'potential_vorticity',
    'boundary_layer_height',
    'geopotential_at_surface', 'land_sea_mask'
]

# 3. Select time, space, variables
# Using method='nearest' for robustness in case slice endpoints don't exactly match coordinates.
print("✂️ Subsetting data...")
ds_subset = ds[vars_to_keep].sel(
    time=slice("1959", "2023"),
    latitude=slice(75, 30), # Assuming latitude is ordered from North to South (e.g., 90 to -90)
    longitude=slice(-25 % 360, 50 % 360), method='nearest' # Ensure correct wraparound and selection
)

# --- Debugging Prints (Very helpful to confirm data before saving) ---
print("\n--- ds_subset Information ---")
print(ds_subset) # This shows the data structure, dimensions, and variables
print("\nds_subset dimensions:")
for dim, size in ds_subset.dims.items():
    print(f"  {dim}: {size}")

if all(size > 0 for size in ds_subset.dims.values()):
    print("\nAll dimensions have size > 0. Proceeding with save.")
else:
    print("\n❌ WARNING: One or more dimensions have a size of 0. The subset is likely empty and will result in a 0-byte file.")
# --- End Debugging Prints ---


🚀 Attempting to load dataset from Google Cloud Storage...


FileNotFoundError: No such file or directory: 'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr'

In [18]:
#!/usr/bin/env python3
"""
Test script for MESA-Net data loading pipeline
Tests WeatherBench2Dataset and identifies issues

Run this first to see what breaks in our data pipeline.
"""

import sys
import traceback
import torch
import xarray as xr
import numpy as np
from typing import List, Dict, Tuple

# Add your mesa_net package to path if needed
# sys.path.append('/path/to/your/mesa_net/')

# Import your classes (adjust imports based on your file structure)
try:
    from src.mesanet.mesanet_dataset import WeatherBench2Dataset
    from src.mesanet.mesanet_datamanager import WeatherBench2DataManager
    print("✅ Imports successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please adjust the import paths based on your file structure")
    sys.exit(1)

class DataLoadingTester:
    """Test data loading step by step"""
    
    def __init__(self):
        self.zarr_path = "gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr"
        
        # Start with minimal variables to test
        self.test_variables = [
            'total_precipitation_6hr',
            '2m_temperature',
            'surface_pressure',
            'mean_sea_level_pressure'
        ]
        
        self.results = {}
    
    def test_1_basic_zarr_access(self):
        """Test 1: Can we access WeatherBench2 zarr directly?"""
        print("\n" + "="*50)
        print("TEST 1: Basic WeatherBench2 Access")
        print("="*50)
        
        try:
            # Try to open the zarr store
            ds = xr.open_zarr(
                self.zarr_path,
                consolidated=True,
                storage_options={"token": "anon"},
                chunks={'time': 10}
            )
            
            print("✅ Successfully opened WeatherBench2 zarr")
            print(f"   Dataset dimensions: {dict(ds.dims)}")
            print(f"   Available variables: {len(list(ds.data_vars.keys()))}")
            print(f"   Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
            
            # Check which of our test variables exist
            available_vars = list(ds.data_vars.keys())
            print(f"\n   Variable availability check:")
            for var in self.test_variables:
                exists = var in available_vars
                status = "✅" if exists else "❌"
                print(f"   {status} {var}")
            
            self.results['zarr_access'] = True
            self.results['available_variables'] = available_vars
            return ds
            
        except Exception as e:
            print(f"❌ Failed to access WeatherBench2: {e}")
            print(f"   Error type: {type(e).__name__}")
            traceback.print_exc()
            self.results['zarr_access'] = False
            return None
    
    def test_2_variable_inspection(self, ds):
        """Test 2: Inspect variables in detail"""
        print("\n" + "="*50)
        print("TEST 2: Variable Inspection")
        print("="*50)
        
        if ds is None:
            print("❌ Skipping - no dataset available")
            return
        
        try:
            # Print first 20 variables with their dimensions
            variables = list(ds.data_vars.keys())[:20]
            print(f"First 20 variables and their dimensions:")
            
            for var in variables:
                dims = ds[var].dims
                shape = ds[var].shape
                print(f"   {var}: {dims} -> {shape}")
            
            # Test specific variables we need
            print(f"\nTesting our required variables:")
            for var in self.test_variables:
                if var in ds.data_vars:
                    var_data = ds[var]
                    print(f"   ✅ {var}: {var_data.dims} -> {var_data.shape}")
                    print(f"      Data type: {var_data.dtype}")
                    print(f"      Min/Max: {float(var_data.min())} / {float(var_data.max())}")
                else:
                    print(f"   ❌ {var}: NOT FOUND")
            
            self.results['variable_inspection'] = True
            
        except Exception as e:
            print(f"❌ Variable inspection failed: {e}")
            traceback.print_exc()
            self.results['variable_inspection'] = False
    
    def test_3_geographic_subsetting(self, ds):
        """Test 3: Geographic subsetting for Europe"""
        print("\n" + "="*50)
        print("TEST 3: Geographic Subsetting")
        print("="*50)
        
        if ds is None:
            print("❌ Skipping - no dataset available")
            return None
        
        try:
            print(f"Original longitude range: {ds.longitude.min().values} to {ds.longitude.max().values}")
            print(f"Original latitude range: {ds.latitude.min().values} to {ds.latitude.max().values}")
            
            # Apply Europe bounds
            ds_europe = ds.where(
                (ds.longitude >= 335) | (ds.longitude <= 50),
                drop=True
            ).sel(latitude=slice(75, 30))
            
            print(f"Europe longitude range: {ds_europe.longitude.min().values} to {ds_europe.longitude.max().values}")
            print(f"Europe latitude range: {ds_europe.latitude.min().values} to {ds_europe.latitude.max().values}")
            print(f"Europe grid shape: lat={len(ds_europe.latitude)}, lon={len(ds_europe.longitude)}")
            
            self.results['geographic_subsetting'] = True
            return ds_europe
            
        except Exception as e:
            print(f"❌ Geographic subsetting failed: {e}")
            traceback.print_exc()
            self.results['geographic_subsetting'] = False
            return None
    
    def test_4_time_subsetting(self, ds):
        """Test 4: Time subsetting and indexing"""
        print("\n" + "="*50)
        print("TEST 4: Time Subsetting")
        print("="*50)
        
        if ds is None:
            print("❌ Skipping - no dataset available")
            return None
        
        try:
            # Test recent time period
            ds_recent = ds.sel(time=slice("2023", "2023"))
            print(f"✅ 2023 data: {len(ds_recent.time)} time steps")
            print(f"   Time range: {ds_recent.time.values[0]} to {ds_recent.time.values[-1]}")
            
            # Test sequence creation indices
            total_time_steps = len(ds_recent.time)
            sequence_length = 12
            forecast_horizon = 4
            
            valid_indices = np.arange(
                sequence_length,
                total_time_steps - forecast_horizon
            )
            
            print(f"   Valid sequence indices: {len(valid_indices)} out of {total_time_steps}")
            
            if len(valid_indices) > 0:
                print(f"   First valid index: {valid_indices[0]}")
                print(f"   Last valid index: {valid_indices[-1]}")
            else:
                print("   ❌ No valid sequences found!")
            
            self.results['time_subsetting'] = True
            return ds_recent, valid_indices
            
        except Exception as e:
            print(f"❌ Time subsetting failed: {e}")
            traceback.print_exc()
            self.results['time_subsetting'] = False
            return None, None
    
    def test_5_single_sample_extraction(self, ds, valid_indices):
        """Test 5: Extract a single training sample"""
        print("\n" + "="*50)
        print("TEST 5: Single Sample Extraction")
        print("="*50)
        
        if ds is None or len(valid_indices) == 0:
            print("❌ Skipping - no dataset or valid indices available")
            return
        
        try:
            sequence_length = 12
            forecast_horizon = 4
            time_idx = valid_indices[0]
            
            print(f"Testing with time index: {time_idx}")
            
            # Input sequence
            input_slice = ds.isel(
                time=slice(time_idx - sequence_length, time_idx)
            )
            print(f"✅ Input slice shape: {input_slice.sizes}")
            print(f"✅ Input shape: {input_slice.shape}")
            for dim, size in input_slice.sizes.items():
                print(f"    {dim}: {size}")
            
            # Target sequence  
            if 'total_precipitation_6hr' in ds.data_vars:
                target_slice = ds['total_precipitation_6hr'].isel(
                    time=slice(time_idx, time_idx + forecast_horizon)
                )
                print(f"✅ Target slice shape: {target_slice.sizes}")
                for dim, size in target_slice.sizes.items():
                    print(f"    {dim}: {size}")
            else:
                print("❌ total_precipitation_6hr not found")
                return
            
            # Test loading the data
            print("Loading data into memory...")
            input_loaded = input_slice.load()
            target_loaded = target_slice.load()
            
            print(f"✅ Successfully loaded sample")
            print(f"   Input time steps: {len(input_loaded.time)}")
            print(f"   Target time steps: {len(target_loaded.time)}")
            
            self.results['sample_extraction'] = True
            return input_loaded, target_loaded
            
        except Exception as e:
            print(f"❌ Sample extraction failed: {e}")
            traceback.print_exc()
            self.results['sample_extraction'] = False
            return None, None
    
    def test_6_tensor_conversion(self, input_data, target_data):
        """Test 6: Convert xarray to PyTorch tensors"""
        print("\n" + "="*50)
        print("TEST 6: Tensor Conversion")
        print("="*50)
        
        if input_data is None or target_data is None:
            print("❌ Skipping - no data available")
            return
        
        try:
            # Test basic tensor conversion for target (single variable)
            print("Testing target tensor conversion...")
            target_array = target_data.values
            target_tensor = torch.tensor(target_array, dtype=torch.float32)
            print(f"✅ Target tensor shape: {target_tensor.shape}")
            print(f"   Data type: {target_tensor.dtype}")
            print(f"   Value range: {target_tensor.min().item():.4f} to {target_tensor.max().item():.4f}")
            
            # Test multi-variable input conversion (this is where we'll likely have issues)
            print("\nTesting input tensor conversion...")
            var_arrays = []
            
            for var in self.test_variables:
                if var in input_data.data_vars:
                    var_data = input_data[var].values
                    print(f"   Processing {var}: shape {var_data.shape}")
                    
                    # Handle different dimensionalities
                    if var_data.ndim == 2:  # (lat, lon) - single time step
                        var_data = var_data[None, ...]  # Add time dimension
                    elif var_data.ndim == 4:  # (time, level, lat, lon) - multi-level
                        # Average over pressure levels
                        var_data = np.mean(var_data, axis=1)
                        print(f"     Averaged over pressure levels: {var_data.shape}")
                    
                    var_arrays.append(var_data)
                    print(f"   ✅ {var}: final shape {var_data.shape}")
                else:
                    print(f"   ❌ {var}: not found in data")
            
            if var_arrays:
                # Stack variables
                stacked_array = np.stack(var_arrays, axis=1)
                input_tensor = torch.tensor(stacked_array, dtype=torch.float32)
                
                print(f"✅ Input tensor shape: {input_tensor.shape}")
                print(f"   Expected format: (time, vars, lat, lon)")
                print(f"   Data type: {input_tensor.dtype}")
                
                self.results['tensor_conversion'] = True
                return input_tensor, target_tensor
            else:
                print("❌ No variables could be processed")
                self.results['tensor_conversion'] = False
                return None, None
            
        except Exception as e:
            print(f"❌ Tensor conversion failed: {e}")
            traceback.print_exc()
            self.results['tensor_conversion'] = False
            return None, None
    
    def test_7_dataset_class(self):
        """Test 7: Our WeatherBench2Dataset class"""
        print("\n" + "="*50)
        print("TEST 7: WeatherBench2Dataset Class")
        print("="*50)
        
        try:
            # Create dataset with minimal configuration
            dataset = WeatherBench2Dataset(
                zarr_path=self.zarr_path,
                variables=self.test_variables,
                time_range=slice("2023", "2023"),  # Just 2023 for testing
                split="train",
                sequence_length=12,
                forecast_horizon=4,
                normalize=False  # Skip normalization for now
            )
            
            print(f"✅ Dataset created successfully")
            print(f"   Dataset length: {len(dataset)}")
            
            # Test getting one sample
            if len(dataset) > 0:
                print("Testing sample retrieval...")
                input_seq, target_seq, geo_features = dataset[0]
                
                print(f"✅ Sample retrieved successfully")
                print(f"   Input sequence shape: {input_seq.shape}")
                print(f"   Target sequence shape: {target_seq.shape}")
                print(f"   Geographic features shape: {geo_features.shape}")
                
                self.results['dataset_class'] = True
            else:
                print("❌ Dataset is empty")
                self.results['dataset_class'] = False
            
        except Exception as e:
            print(f"❌ Dataset class test failed: {e}")
            traceback.print_exc()
            self.results['dataset_class'] = False
    
    def run_all_tests(self):
        """Run all tests in sequence"""
        print("🚀 Starting MESA-Net Data Loading Tests")
        print("="*60)
        
        # Test 1: Basic access
        ds = self.test_1_basic_zarr_access()
        
        # Test 2: Variable inspection
        self.test_2_variable_inspection(ds)
        
        # Test 3: Geographic subsetting
        ds_europe = self.test_3_geographic_subsetting(ds)
        
        # Test 4: Time subsetting
        ds_recent, valid_indices = self.test_4_time_subsetting(ds_europe)
        
        # Test 5: Sample extraction
        input_data, target_data = self.test_5_single_sample_extraction(ds_recent, valid_indices)
        
        # Test 6: Tensor conversion
        input_tensor, target_tensor = self.test_6_tensor_conversion(input_data, target_data)
        
        # Test 7: Dataset class
        self.test_7_dataset_class()
        
        # Summary
        self.print_summary()
    
    def print_summary(self):
        """Print test results summary"""
        print("\n" + "="*60)
        print("🏁 TEST RESULTS SUMMARY")
        print("="*60)
        
        tests = [
            ('zarr_access', 'WeatherBench2 Access'),
            ('variable_inspection', 'Variable Inspection'),
            ('geographic_subsetting', 'Geographic Subsetting'),
            ('time_subsetting', 'Time Subsetting'),
            ('sample_extraction', 'Sample Extraction'),
            ('tensor_conversion', 'Tensor Conversion'),
            ('dataset_class', 'Dataset Class'),
        ]
        
        passed = 0
        total = len(tests)
        
        for test_key, test_name in tests:
            if test_key in self.results:
                status = "✅ PASS" if self.results[test_key] else "❌ FAIL"
                if self.results[test_key]:
                    passed += 1
            else:
                status = "⏭️ SKIP"
            
            print(f"{status} {test_name}")
        
        print(f"\nOverall: {passed}/{total} tests passed")
        
        if passed == total:
            print("🎉 All tests passed! Data pipeline is working.")
        else:
            print("🔧 Some tests failed. Check the errors above and fix issues.")
            
        return passed == total

if __name__ == "__main__":
    tester = DataLoadingTester()
    success = tester.run_all_tests()
    
    if success:
        print("\n✅ Ready to proceed to next step!")
    else:
        print("\n❌ Fix data loading issues before proceeding.")
        sys.exit(1)

✅ Imports successful
🚀 Starting MESA-Net Data Loading Tests

TEST 1: Basic WeatherBench2 Access
✅ Successfully opened WeatherBench2 zarr
   Dataset dimensions: {'time': 93544, 'latitude': 721, 'longitude': 1440, 'level': 13}
   Available variables: 62
   Time range: 1959-01-01T00:00:00.000000000 to 2023-01-10T18:00:00.000000000

   Variable availability check:
   ✅ total_precipitation_6hr
   ✅ 2m_temperature
   ✅ surface_pressure
   ✅ mean_sea_level_pressure

TEST 2: Variable Inspection
First 20 variables and their dimensions:
   10m_u_component_of_wind: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
   10m_v_component_of_wind: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
   10m_wind_speed: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
   2m_dewpoint_temperature: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
   2m_temperature: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
   above_ground: ('time', 'level', 'latitude', 'longitude') -> (9

C:\Users\peera\AppData\Local\Temp\ipykernel_9572\3145462854.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"   Dataset dimensions: {dict(ds.dims)}")


      Min/Max: -2.2351741790771484e-08 / 0.4034213125705719
   ✅ 2m_temperature: ('time', 'latitude', 'longitude') -> (93544, 721, 1440)
      Data type: float32


KeyboardInterrupt: 